# Architecture Diagrams

Standalone cells to regenerate all architecture diagrams for the report.
Each cell saves its PNG to `results/architecture_diagrams/`.
Run any cell independently — no project config required.


In [ ]:
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from pathlib import Path

OUT_DIR = Path('/home/littl/ECE247A_Final_Project/JZ/results/architecture_diagrams')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Shared colour palette — same across all diagrams
C = dict(
    input ='#D6EAF8',   # light blue – raw EEG input
    spec  ='#5DADE2',   # darker blue – spectrogram input
    conv  ='#A9DFBF',   # green  – conv / patch embed
    gru   ='#FAD7A0',   # orange – backbone / recurrent
    attn  ='#D7BDE2',   # purple – norm + pooling
    head  ='#F9E79F',   # yellow – classification head
    out   ='#F1948A',   # red    – output
)

In [ ]:
# ── EfficientNetV2B2 spectrogram baseline architecture diagram ─────────────────
# 5-box layout matching the Swin-T diagram style:
#   Input → EfficientNetV2B2 Backbone → GlobalAvg Pool → Dense Head → Output
OUT_PATH = OUT_DIR / 'efficientnet_v2b2_architecture_diagram.png'

FW, FH = 15, 2.9
fig, ax = plt.subplots(figsize=(FW, FH))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis('off')

N_BOXES = 5
BW   = 0.130
BH   = 0.40
GAP  = 0.040
CY   = 0.65

MARGIN = (1.0 - N_BOXES * BW - (N_BOXES - 1) * GAP) / 2
CXS    = [MARGIN + BW / 2 + i * (BW + GAP) for i in range(N_BOXES)]

SHAPE_Y  = CY - BH / 2 - 0.065
LEGEND_Y = SHAPE_Y - 0.12

def _box(i, lines, color='white'):
    cx = CXS[i]
    lx, ly = cx - BW / 2, CY - BH / 2
    ax.add_patch(FancyBboxPatch(
        (lx, ly), BW, BH,
        boxstyle='round,pad=0.012',
        facecolor=color, edgecolor='#555555', linewidth=1.2, zorder=2,
    ))
    n = len(lines)
    for j, txt in enumerate(lines):
        rel = (j + 0.5) / n
        y   = CY + BH * (0.5 - rel) * 0.78
        fs  = 12.0 if j == 0 else 10.5
        ax.text(cx, y, txt, ha='center', va='center',
                fontsize=fs, fontweight='bold' if j == 0 else 'normal',
                color='#111111' if j == 0 else '#333333', zorder=3)

def _arrow(i_from, i_to, shape_txt=None):
    inset = GAP * 0.15
    x0  = CXS[i_from] + BW / 2 + inset
    x1  = CXS[i_to]   - BW / 2 - inset
    mid = (x0 + x1) / 2
    ax.annotate('', xy=(x1, CY), xytext=(x0, CY),
                arrowprops=dict(arrowstyle='->', color='#555555', lw=1.3), zorder=1)
    if shape_txt:
        ax.text(mid, SHAPE_Y, shape_txt, ha='center', va='center',
                fontsize=9.0, color='#666666', style='italic', zorder=3)

# ── blocks ─────────────────────────────────────────────────────────────────────
_box(0, ['Spectrogram',        '3 channels',       'H × W',             '(float32)'],      C['spec'])
_box(1, ['EfficientNetV2B2',   'ImageNet init',     '8.77M params',      'multi-scale feat'], C['conv'])
_box(2, ['GlobalAvg',          'Pooling',           '',                  '(1408,)'],        C['attn'])
_box(3, ['Dense Head',         '1408 → 6',          '~8.5K params',      'softmax'],        C['head'])
_box(4, ['Output',             '6-class',           'prob. dist.',       ''],               C['out'])

# ── arrows ─────────────────────────────────────────────────────────────────────
_arrow(0, 1, '(3, H, W)')
_arrow(1, 2, "(H', W', 1408)")
_arrow(2, 3, '(1408,)')
_arrow(3, 4, '(6,)')

# ── legend ─────────────────────────────────────────────────────────────────────
ax.legend(handles=[
    mpatches.Patch(facecolor=C['spec'],  edgecolor='#555555', label='Spectrogram input'),
    mpatches.Patch(facecolor=C['conv'],  edgecolor='#555555', label='EfficientNetV2B2 backbone'),
    mpatches.Patch(facecolor=C['attn'],  edgecolor='#555555', label='Global avg pooling'),
    mpatches.Patch(facecolor=C['head'],  edgecolor='#555555', label='Dense head'),
    mpatches.Patch(facecolor=C['out'],   edgecolor='#555555', label='Output'),
], loc='lower center', fontsize=9.5, framealpha=0.0, edgecolor='none',
   bbox_to_anchor=(0.5, LEGEND_Y),
   ncol=5, columnspacing=0.7, handlelength=1.2, handletextpad=0.4, borderpad=0.2)

# ── title ──────────────────────────────────────────────────────────────────────
ax.text(0.50, 0.975,
        'Spectrogram  →  EfficientNetV2B2 Backbone  →  Dense Head  '
        '→  6-class probability distribution',
        ha='center', va='center', fontsize=13.0, fontweight='bold')

fig.tight_layout(pad=0.3)
fig.savefig(OUT_PATH, dpi=200, bbox_inches='tight', facecolor='white')
print(f'Saved: {OUT_PATH}')
plt.show()
plt.close(fig)


In [ ]:
# ── Spectrogram-GRU model architecture diagram ────────────────────────────────
# 6-box layout:
#   Input EEG → STFT ×2 → EfficientNetV2 → Bi-GRU + Attn → Class. Head → Output
# Auxiliary stats branch (32-dim) concatenated before head is noted in the Head box.
OUT_PATH = OUT_DIR / 'spectrogram_gru_architecture_diagram.png'

FW, FH = 15, 2.9
fig, ax = plt.subplots(figsize=(FW, FH))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis('off')

N_BOXES = 6
BW   = 0.110
BH   = 0.40
GAP  = 0.030
CY   = 0.65

MARGIN = (1.0 - N_BOXES * BW - (N_BOXES - 1) * GAP) / 2
CXS    = [MARGIN + BW / 2 + i * (BW + GAP) for i in range(N_BOXES)]

SHAPE_Y  = CY - BH / 2 - 0.065
LEGEND_Y = SHAPE_Y - 0.12

def _box(i, lines, color='white'):
    cx = CXS[i]
    lx, ly = cx - BW / 2, CY - BH / 2
    ax.add_patch(FancyBboxPatch(
        (lx, ly), BW, BH,
        boxstyle='round,pad=0.012',
        facecolor=color, edgecolor='#555555', linewidth=1.2, zorder=2,
    ))
    n = len(lines)
    for j, txt in enumerate(lines):
        rel = (j + 0.5) / n
        y   = CY + BH * (0.5 - rel) * 0.78
        fs  = 12.0 if j == 0 else 10.5
        ax.text(cx, y, txt, ha='center', va='center',
                fontsize=fs, fontweight='bold' if j == 0 else 'normal',
                color='#111111' if j == 0 else '#333333', zorder=3)

def _arrow(i_from, i_to, shape_txt=None):
    inset = GAP * 0.15
    x0  = CXS[i_from] + BW / 2 + inset
    x1  = CXS[i_to]   - BW / 2 - inset
    mid = (x0 + x1) / 2
    ax.annotate('', xy=(x1, CY), xytext=(x0, CY),
                arrowprops=dict(arrowstyle='->', color='#555555', lw=1.3), zorder=1)
    if shape_txt:
        ax.text(mid, SHAPE_Y, shape_txt, ha='center', va='center',
                fontsize=9.0, color='#666666', style='italic', zorder=3)

# ── blocks ─────────────────────────────────────────────────────────────────────
_box(0, ['Input EEG',        '16 channels',       'raw signal',        '(200 Hz)'],         C['input'])
_box(1, ['STFT ×2',          'Wide [257 × 79]',   'Zoom [257 × 63]',   '~526K params'],      C['conv'])
_box(2, ['EfficientNetV2',   'Backbone',          '8.39M params',      '→ [16, 416, 40]'],  C['conv'])
_box(3, ['Bi-GRU',           '+ Attention',       '1.9M params',       '→ (512,)'],         C['gru'])
_box(4, ['Class. Head',      '⊕ Aux Stats (32)',  '544 → 128 → 6',     '~70.5K params'],    C['head'])
_box(5, ['Output',           '6-class',           'prob. dist.',       ''],                  C['out'])

# ── arrows ─────────────────────────────────────────────────────────────────────
_arrow(0, 1, 'raw EEG')
_arrow(1, 2, '[16, 257, ·]')
_arrow(2, 3, '[16, 416, 40]')
_arrow(3, 4, '(512,)')
_arrow(4, 5, '(6,)')

# ── legend ─────────────────────────────────────────────────────────────────────
ax.legend(handles=[
    mpatches.Patch(facecolor=C['input'], edgecolor='#555555', label='Raw EEG input'),
    mpatches.Patch(facecolor=C['conv'],  edgecolor='#555555', label='STFT spectrogram'),
    mpatches.Patch(facecolor=C['conv'],  edgecolor='#555555', label='EfficientNetV2 backbone'),
    mpatches.Patch(facecolor=C['gru'],   edgecolor='#555555', label='Bi-GRU + attention'),
    mpatches.Patch(facecolor=C['head'],  edgecolor='#555555', label='Classification head'),
    mpatches.Patch(facecolor=C['out'],   edgecolor='#555555', label='Output'),
], loc='lower center', fontsize=9.5, framealpha=0.0, edgecolor='none',
   bbox_to_anchor=(0.5, LEGEND_Y),
   ncol=6, columnspacing=0.5, handlelength=1.2, handletextpad=0.4, borderpad=0.2)

# ── title ──────────────────────────────────────────────────────────────────────
ax.text(0.50, 0.975,
        'Raw EEG  →  STFT ×2  →  EfficientNetV2 + Bi-GRU  →  6-class probability distribution',
        ha='center', va='center', fontsize=13.0, fontweight='bold')

fig.tight_layout(pad=0.3)
fig.savefig(OUT_PATH, dpi=200, bbox_inches='tight', facecolor='white')
print(f'Saved: {OUT_PATH}')
plt.show()
plt.close(fig)


## Conv1D + BiGRU (Raw EEG)


In [ ]:
# ── Conv1D + BiGRU raw EEG architecture diagram ──────────────────────────────
OUT_PATH = OUT_DIR / 'conv1d_bigru_architecture_diagram.png'

# ── figure setup ──────────────────────────────────────────────────────────────
FW, FH = 15, 2.9
fig, ax = plt.subplots(figsize=(FW, FH))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis('off')

# ── geometry ──────────────────────────────────────────────────────────────────
N_BOXES = 6
BW   = 0.110   # box width (normalised) — wider with fewer boxes
BH   = 0.40    # box height (normalised)
GAP  = 0.032   # gap between box edge and arrow tip
CY   = 0.65    # vertical centre of every box

MARGIN = (1.0 - N_BOXES * BW - (N_BOXES - 1) * GAP) / 2
CXS = [MARGIN + BW / 2 + i * (BW + GAP) for i in range(N_BOXES)]

SHAPE_Y  = CY - BH / 2 - 0.065
LEGEND_Y = SHAPE_Y - 0.12

# ── helper functions ──────────────────────────────────────────────────────────
def block(i, lines, color='white'):
    cx = CXS[i]
    lx, ly = cx - BW / 2, CY - BH / 2
    ax.add_patch(FancyBboxPatch(
        (lx, ly), BW, BH,
        boxstyle='round,pad=0.012',
        facecolor=color, edgecolor='#777777', linewidth=1.0, zorder=2,
    ))
    n = len(lines)
    for j, txt in enumerate(lines):
        rel = (j + 0.5) / n
        y   = CY + BH * (0.5 - rel) * 0.78
        fs  = 12.0 if j == 0 else 10.5
        ax.text(cx, y, txt, ha='center', va='center',
                fontsize=fs, fontweight='bold' if j == 0 else 'normal',
                color='#111111' if j == 0 else '#333333', zorder=3)

def arrow(i_from, i_to, shape_txt=None):
    inset = GAP * 0.15
    x0  = CXS[i_from] + BW / 2 + inset
    x1  = CXS[i_to]   - BW / 2 - inset
    mid = (x0 + x1) / 2
    ax.annotate('', xy=(x1, CY), xytext=(x0, CY),
                arrowprops=dict(arrowstyle='->', color='#555555', lw=1.3), zorder=1)
    if shape_txt:
        ax.text(mid, SHAPE_Y, shape_txt, ha='center', va='center',
                fontsize=9.0, color='#666666', style='italic', zorder=3)

# ── blocks ────────────────────────────────────────────────────────────────────
block(0, ['Input EEG',    '16 channels',  '5 000 steps',  '(100 Hz)'],     C['input'])
block(1, ['4× ConvBlock', 'k = 7, 5, 5, 3', 'stride 2',  '16 → 192 ch'],  C['conv'])
block(2, ['Bi-GRU',       '1 layer',      'hidden = 160', 'bidir.'],        C['gru'])
block(3, ['Attention',    'Pooling',      'softmax(t)',    'sum'],           C['attn'])
block(4, ['Class. Head',  '320 → 128',    'SiLU',         '128 → 6'],       C['head'])
block(5, ['Softmax',      'Output',       '6-class',      'prob. dist.'],    C['out'])

# ── arrows + shape labels ─────────────────────────────────────────────────────
arrow(0, 1, '(16, 5000)')
arrow(1, 2, '(192, 312)')
arrow(2, 3, '(312, 320)')
arrow(3, 4, '(320,)')
arrow(4, 5, '(6,)')

# ── legend ────────────────────────────────────────────────────────────────────
ax.legend(handles=[
    mpatches.Patch(facecolor=C['input'], edgecolor='#777777', label='Input'),
    mpatches.Patch(facecolor=C['conv'],  edgecolor='#777777', label='Conv1D block (×4)'),
    mpatches.Patch(facecolor=C['gru'],   edgecolor='#777777', label='Bidirectional GRU'),
    mpatches.Patch(facecolor=C['attn'],  edgecolor='#777777', label='Attention pooling'),
    mpatches.Patch(facecolor=C['head'],  edgecolor='#777777', label='Classification head'),
    mpatches.Patch(facecolor=C['out'],   edgecolor='#777777', label='Output'),
], loc='lower center', fontsize=9.5, framealpha=0.0, edgecolor='none',
   bbox_to_anchor=(0.5, LEGEND_Y),
   ncol=6, columnspacing=0.7, handlelength=1.0, handletextpad=0.4, borderpad=0.2)


# ── title ─────────────────────────────────────────────────────────────────────
ax.text(0.50, 0.975,
        'Raw EEG  →  Conv1D + BiGRU + Attention  →  6-class probability distribution',
        ha='center', va='center', fontsize=13.0, fontweight='bold')

fig.tight_layout(pad=0.3)
fig.savefig(OUT_PATH, dpi=200, bbox_inches='tight', facecolor='white')
print(f'Saved: {OUT_PATH}')
plt.show()
plt.close(fig)


## Swin-T (Spectrogram)


In [ ]:
# ── Swin-T spectrogram architecture diagram ────────────────────────────────────
OUT_PATH = OUT_DIR / 'vit_swin_architecture_diagram.png'


# ── figure setup ──────────────────────────────────────────────────────────────
FW, FH = 15, 2.9
fig, ax = plt.subplots(figsize=(FW, FH))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis('off')

# ── geometry ──────────────────────────────────────────────────────────────────
N_BOXES = 5
BW   = 0.130
BH   = 0.40
GAP  = 0.040
CY   = 0.65   # raised — no brackets above boxes

MARGIN = (1.0 - N_BOXES * BW - (N_BOXES - 1) * GAP) / 2
CXS    = [MARGIN + BW / 2 + i * (BW + GAP) for i in range(N_BOXES)]

SHAPE_Y  = CY - BH / 2 - 0.065
LEGEND_Y = SHAPE_Y - 0.12

# ── helpers ───────────────────────────────────────────────────────────────────
def _box(i, lines, color='white'):
    cx = CXS[i]
    lx, ly = cx - BW / 2, CY - BH / 2
    ax.add_patch(FancyBboxPatch(
        (lx, ly), BW, BH,
        boxstyle='round,pad=0.012',
        facecolor=color, edgecolor='#555555', linewidth=1.2,
        zorder=2,
    ))
    n = len(lines)
    for j, txt in enumerate(lines):
        rel = (j + 0.5) / n
        y   = CY + BH * (0.5 - rel) * 0.78
        fs  = 12.0 if j == 0 else 10.5
        ax.text(cx, y, txt, ha='center', va='center',
                fontsize=fs, fontweight='bold' if j == 0 else 'normal',
                color='#111111' if j == 0 else '#333333', zorder=3)

def _arrow(i_from, i_to, shape_txt=None):
    inset = GAP * 0.15
    x0  = CXS[i_from] + BW / 2 + inset
    x1  = CXS[i_to]   - BW / 2 - inset
    mid = (x0 + x1) / 2
    ax.annotate('', xy=(x1, CY), xytext=(x0, CY),
                arrowprops=dict(arrowstyle='->', color='#555555', lw=1.3), zorder=1)
    if shape_txt:
        ax.text(mid, SHAPE_Y, shape_txt, ha='center', va='center',
                fontsize=9.0, color='#666666', style='italic', zorder=3)

# ── blocks ────────────────────────────────────────────────────────────────────
_box(0, ['Spectrogram',     '3 channels',       '224 × 224',        '(float32)'],      C['input'])
_box(1, ['Swin-T Backbone', 'patch embed',       '+ 4 Swin stages',  'ImageNet init'],  C['gru'])
_box(2, ['LayerNorm',       'GlobalAvg',         'Pool',             '(768,)'],         C['attn'])
_box(3, ['Head',            '768 → 256',         'SiLU  Drop(0.3)',  '256 → 6'],        C['head'])
_box(4, ['Softmax',         'Output',            '6-class',          'prob. dist.'],    C['out'])

# ── arrows with tensor shapes ─────────────────────────────────────────────────
_arrow(0, 1, '(3, 224, 224)')
_arrow(1, 2, '(7×7, 768ch)')
_arrow(2, 3, '(768,)')
_arrow(3, 4, '(6,)')

# ── legend ────────────────────────────────────────────────────────────────────
ax.legend(handles=[
    mpatches.Patch(facecolor=C['spec'],  edgecolor='#555555', label='Spectrogram input'),
    mpatches.Patch(facecolor=C['gru'],   edgecolor='#555555', label='Swin-T backbone'),
    mpatches.Patch(facecolor=C['attn'],  edgecolor='#555555', label='Norm + Pooling'),
    mpatches.Patch(facecolor=C['head'],  edgecolor='#555555', label='Classification head'),
    mpatches.Patch(facecolor=C['out'],   edgecolor='#555555', label='Output'),
], loc='lower center', fontsize=9.5, framealpha=0.0, edgecolor='none',
   bbox_to_anchor=(0.5, LEGEND_Y),
   ncol=5, columnspacing=0.7, handlelength=1.2, handletextpad=0.4, borderpad=0.2)

# ── title ─────────────────────────────────────────────────────────────────────
ax.text(0.50, 0.975,
        'Kaggle Spectrogram  →  Swin-T Backbone  →  Classification Head  '
        '→  6-class probability distribution',
        ha='center', va='center', fontsize=13.0, fontweight='bold')

fig.tight_layout(pad=0.3)
fig.savefig(OUT_PATH, dpi=200, bbox_inches='tight', facecolor='white')
print(f'Saved: {OUT_PATH}')
plt.show()
plt.close(fig)


## EfficientNet-B0 Spectrogram CNN


In [ ]:
# ── EfficientNet-B0 spectrogram CNN architecture diagram ──────────────────────
# 5-box layout:
#   Spectrogram → EfficientNet-B0 Backbone → GlobalAvg Pool → Head → Output
OUT_PATH = OUT_DIR / 'efficientnet_b0_cnn_architecture_diagram.png'

FW, FH = 15, 2.9
fig, ax = plt.subplots(figsize=(FW, FH))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis('off')

N_BOXES = 5
BW   = 0.130
BH   = 0.40
GAP  = 0.040
CY   = 0.65

MARGIN = (1.0 - N_BOXES * BW - (N_BOXES - 1) * GAP) / 2
CXS    = [MARGIN + BW / 2 + i * (BW + GAP) for i in range(N_BOXES)]

SHAPE_Y  = CY - BH / 2 - 0.065
LEGEND_Y = SHAPE_Y - 0.12

def _box(i, lines, color='white'):
    cx = CXS[i]
    lx, ly = cx - BW / 2, CY - BH / 2
    ax.add_patch(FancyBboxPatch(
        (lx, ly), BW, BH,
        boxstyle='round,pad=0.012',
        facecolor=color, edgecolor='#555555', linewidth=1.2, zorder=2,
    ))
    n = len(lines)
    for j, txt in enumerate(lines):
        rel = (j + 0.5) / n
        y   = CY + BH * (0.5 - rel) * 0.78
        fs  = 12.0 if j == 0 else 10.5
        ax.text(cx, y, txt, ha='center', va='center',
                fontsize=fs, fontweight='bold' if j == 0 else 'normal',
                color='#111111' if j == 0 else '#333333', zorder=3)

def _arrow(i_from, i_to, shape_txt=None):
    inset = GAP * 0.15
    x0  = CXS[i_from] + BW / 2 + inset
    x1  = CXS[i_to]   - BW / 2 - inset
    mid = (x0 + x1) / 2
    ax.annotate('', xy=(x1, CY), xytext=(x0, CY),
                arrowprops=dict(arrowstyle='->', color='#555555', lw=1.3), zorder=1)
    if shape_txt:
        ax.text(mid, SHAPE_Y, shape_txt, ha='center', va='center',
                fontsize=9.0, color='#666666', style='italic', zorder=3)

# ── blocks ─────────────────────────────────────────────────────────────────────
_box(0, ['Spectrogram',      '3 channels',      '256 × 256',        '(float32)'],      C['spec'])
_box(1, ['EfficientNet-B0',  '7 MBConv stages', '4.0M params',      '→ (1280, 8, 8)'], C['conv'])
_box(2, ['GlobalAvg',        'Pooling',         '',                 '(1280,)'],         C['attn'])
_box(3, ['Head',             'Dropout',         '1280 → 6',         '~7.7K params'],    C['head'])
_box(4, ['Softmax',          'Output',          '6-class',          'prob. dist.'],     C['out'])

# ── arrows ─────────────────────────────────────────────────────────────────────
_arrow(0, 1, '(3, 256, 256)')
_arrow(1, 2, '(1280, 8, 8)')
_arrow(2, 3, '(1280,)')
_arrow(3, 4, '(6,)')

# ── legend ─────────────────────────────────────────────────────────────────────
ax.legend(handles=[
    mpatches.Patch(facecolor=C['spec'],  edgecolor='#555555', label='Spectrogram input'),
    mpatches.Patch(facecolor=C['conv'],  edgecolor='#555555', label='EfficientNet-B0 backbone'),
    mpatches.Patch(facecolor=C['attn'],  edgecolor='#555555', label='Global avg pooling'),
    mpatches.Patch(facecolor=C['head'],  edgecolor='#555555', label='Classification head'),
    mpatches.Patch(facecolor=C['out'],   edgecolor='#555555', label='Output'),
], loc='lower center', fontsize=9.5, framealpha=0.0, edgecolor='none',
   bbox_to_anchor=(0.5, LEGEND_Y),
   ncol=5, columnspacing=0.7, handlelength=1.2, handletextpad=0.4, borderpad=0.2)

# ── title ──────────────────────────────────────────────────────────────────────
ax.text(0.50, 0.975,
        'Spectrogram  →  EfficientNet-B0 Backbone  →  Classification Head  '
        '→  6-class probability distribution',
        ha='center', va='center', fontsize=13.0, fontweight='bold')

fig.tight_layout(pad=0.3)
fig.savefig(OUT_PATH, dpi=200, bbox_inches='tight', facecolor='white')
print(f'Saved: {OUT_PATH}')
plt.show()
plt.close(fig)


## EfficientNet-B0 + Transformer (Spectrogram)


In [ ]:
# ── EfficientNet-B0 + Transformer spectrogram model architecture diagram ───────
# 7-box layout:
#   Spectrogram → EfficientNet → Token Proj → Transformer → LayerNorm+Pool → Head → Output
OUT_PATH = OUT_DIR / 'efficientnet_transformer_architecture_diagram.png'

FW, FH = 15, 2.9
fig, ax = plt.subplots(figsize=(FW, FH))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis('off')

N_BOXES = 7
BW   = 0.090
BH   = 0.40
GAP  = 0.022
CY   = 0.65

MARGIN = (1.0 - N_BOXES * BW - (N_BOXES - 1) * GAP) / 2
CXS    = [MARGIN + BW / 2 + i * (BW + GAP) for i in range(N_BOXES)]

SHAPE_Y  = CY - BH / 2 - 0.065
LEGEND_Y = SHAPE_Y - 0.12

def _box(i, lines, color='white'):
    cx = CXS[i]
    lx, ly = cx - BW / 2, CY - BH / 2
    ax.add_patch(FancyBboxPatch(
        (lx, ly), BW, BH,
        boxstyle='round,pad=0.012',
        facecolor=color, edgecolor='#555555', linewidth=1.2, zorder=2,
    ))
    n = len(lines)
    for j, txt in enumerate(lines):
        rel = (j + 0.5) / n
        y   = CY + BH * (0.5 - rel) * 0.78
        fs  = 10.5 if j == 0 else 9.0
        ax.text(cx, y, txt, ha='center', va='center',
                fontsize=fs, fontweight='bold' if j == 0 else 'normal',
                color='#111111' if j == 0 else '#333333', zorder=3)

def _arrow(i_from, i_to, shape_txt=None):
    inset = GAP * 0.15
    x0  = CXS[i_from] + BW / 2 + inset
    x1  = CXS[i_to]   - BW / 2 - inset
    mid = (x0 + x1) / 2
    ax.annotate('', xy=(x1, CY), xytext=(x0, CY),
                arrowprops=dict(arrowstyle='->', color='#555555', lw=1.3), zorder=1)
    if shape_txt:
        ax.text(mid, SHAPE_Y, shape_txt, ha='center', va='center',
                fontsize=8.0, color='#666666', style='italic', zorder=3)

# ── blocks ─────────────────────────────────────────────────────────────────────
_box(0, ['Spectrogram',    '3 channels',    '256 × 256',    '(float32)'],     C['spec'])
_box(1, ['EfficientNet-B0','4.0M params',   '→ (1280, 8, 8)','ImageNet init'],C['conv'])
_box(2, ['Token Proj.',    'Linear',        '1280 → 256',   '64 tokens'],     C['conv'])
_box(3, ['Transformer',    'Encoder',       '2 layers',     '1.58M params'],  C['gru'])
_box(4, ['LayerNorm',      '+ Pool',        '',             '(256,)'],        C['attn'])
_box(5, ['Head',           'Dropout',       '256 → 6',      '~1.5K params'],  C['head'])
_box(6, ['Softmax',        'Output',        '6-class',      'prob. dist.'],   C['out'])

# ── arrows ─────────────────────────────────────────────────────────────────────
_arrow(0, 1, '(3, 256, 256)')
_arrow(1, 2, '(1280, 8, 8)')
_arrow(2, 3, '(64, 256)')
_arrow(3, 4, '(64, 256)')
_arrow(4, 5, '(256,)')
_arrow(5, 6, '(6,)')

# ── legend ─────────────────────────────────────────────────────────────────────
ax.legend(handles=[
    mpatches.Patch(facecolor=C['spec'],  edgecolor='#555555', label='Spectrogram input'),
    mpatches.Patch(facecolor=C['conv'],  edgecolor='#555555', label='CNN / linear projection'),
    mpatches.Patch(facecolor=C['gru'],   edgecolor='#555555', label='Transformer encoder'),
    mpatches.Patch(facecolor=C['attn'],  edgecolor='#555555', label='Norm + pooling'),
    mpatches.Patch(facecolor=C['head'],  edgecolor='#555555', label='Classification head'),
    mpatches.Patch(facecolor=C['out'],   edgecolor='#555555', label='Output'),
], loc='lower center', fontsize=9.5, framealpha=0.0, edgecolor='none',
   bbox_to_anchor=(0.5, LEGEND_Y),
   ncol=6, columnspacing=0.5, handlelength=1.2, handletextpad=0.4, borderpad=0.2)

# ── title ──────────────────────────────────────────────────────────────────────
ax.text(0.50, 0.975,
        'Spectrogram  →  EfficientNet-B0  →  Transformer Encoder  '
        '→  6-class probability distribution',
        ha='center', va='center', fontsize=13.0, fontweight='bold')

fig.tight_layout(pad=0.3)
fig.savefig(OUT_PATH, dpi=200, bbox_inches='tight', facecolor='white')
print(f'Saved: {OUT_PATH}')
plt.show()
plt.close(fig)


## Scalogram-based CNN (CWT + EfficientNet)


In [ ]:
# ── Scalogram-based CNN (CWT + EfficientNet) architecture diagram ──────────────
# 6-box layout:
#   Input EEG → CWT (frozen) → EfficientNet → GlobalAvg Pool → Head → Output
# Dashed border on CWT = non-trainable (frozen wavelet filters).
OUT_PATH = OUT_DIR / 'scalogram_cnn_architecture_diagram.png'

FW, FH = 15, 2.9
fig, ax = plt.subplots(figsize=(FW, FH))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis('off')

N_BOXES = 6
BW   = 0.110
BH   = 0.40
GAP  = 0.030
CY   = 0.65

MARGIN = (1.0 - N_BOXES * BW - (N_BOXES - 1) * GAP) / 2
CXS    = [MARGIN + BW / 2 + i * (BW + GAP) for i in range(N_BOXES)]

SHAPE_Y  = CY - BH / 2 - 0.065
LEGEND_Y = SHAPE_Y - 0.12

def _box(i, lines, color='white', frozen=False):
    cx = CXS[i]
    lx, ly = cx - BW / 2, CY - BH / 2
    ls = (0, (4, 2)) if frozen else 'solid'
    ax.add_patch(FancyBboxPatch(
        (lx, ly), BW, BH,
        boxstyle='round,pad=0.012',
        facecolor=color, edgecolor='#555555', linewidth=1.2,
        linestyle=ls, zorder=2,
    ))
    n = len(lines)
    for j, txt in enumerate(lines):
        rel = (j + 0.5) / n
        y   = CY + BH * (0.5 - rel) * 0.78
        fs  = 12.0 if j == 0 else 10.5
        ax.text(cx, y, txt, ha='center', va='center',
                fontsize=fs, fontweight='bold' if j == 0 else 'normal',
                color='#111111' if j == 0 else '#333333', zorder=3)

def _arrow(i_from, i_to, shape_txt=None):
    inset = GAP * 0.15
    x0  = CXS[i_from] + BW / 2 + inset
    x1  = CXS[i_to]   - BW / 2 - inset
    mid = (x0 + x1) / 2
    ax.annotate('', xy=(x1, CY), xytext=(x0, CY),
                arrowprops=dict(arrowstyle='->', color='#555555', lw=1.3), zorder=1)
    if shape_txt:
        ax.text(mid, SHAPE_Y, shape_txt, ha='center', va='center',
                fontsize=9.0, color='#666666', style='italic', zorder=3)

# ── blocks ─────────────────────────────────────────────────────────────────────
_box(0, ['Input EEG',      '8 channels',     'raw signal',     ''],              C['input'])
_box(1, ['CWT',            'Scalogram',       '1.23M params',   '(frozen)'],     C['conv'], frozen=True)
_box(2, ['EfficientNet',   'Backbone',        '4.0M params',    '→ (1280,4,79)'],C['conv'])
_box(3, ['GlobalAvg',      'Pooling',         '',               '(1280,)'],      C['attn'])
_box(4, ['Head',           'Linear',          '1280 → 6',       '~7.7K params'], C['head'])
_box(5, ['Softmax',        'Output',          '6-class',        'prob. dist.'],  C['out'])

# ── arrows ─────────────────────────────────────────────────────────────────────
_arrow(0, 1, '(8, T)')
_arrow(1, 2, '(8, 128, 2500)')
_arrow(2, 3, '(1280, 4, 79)')
_arrow(3, 4, '(1280,)')
_arrow(4, 5, '(6,)')

# ── legend ─────────────────────────────────────────────────────────────────────
_frozen_patch = mpatches.Patch(facecolor=C['conv'], edgecolor='#555555',
                                linewidth=1.2, linestyle=(0, (4, 2)),
                                label='Dashed = frozen (non-trainable)')
ax.legend(handles=[
    mpatches.Patch(facecolor=C['input'], edgecolor='#555555', label='Raw EEG input'),
    mpatches.Patch(facecolor=C['conv'],  edgecolor='#555555', label='CWT / EfficientNet backbone'),
    mpatches.Patch(facecolor=C['attn'],  edgecolor='#555555', label='Global avg pooling'),
    mpatches.Patch(facecolor=C['head'],  edgecolor='#555555', label='Classification head'),
    mpatches.Patch(facecolor=C['out'],   edgecolor='#555555', label='Output'),
    _frozen_patch,
], loc='lower center', fontsize=9.5, framealpha=0.0, edgecolor='none',
   bbox_to_anchor=(0.5, LEGEND_Y),
   ncol=6, columnspacing=0.5, handlelength=1.2, handletextpad=0.4, borderpad=0.2)

# ── title ──────────────────────────────────────────────────────────────────────
ax.text(0.50, 0.975,
        'Raw EEG  →  CWT Scalogram  →  EfficientNet Backbone  '
        '→  6-class probability distribution',
        ha='center', va='center', fontsize=13.0, fontweight='bold')

fig.tight_layout(pad=0.3)
fig.savefig(OUT_PATH, dpi=200, bbox_inches='tight', facecolor='white')
print(f'Saved: {OUT_PATH}')
plt.show()
plt.close(fig)
